# Créer un ensemble de données d'entraînement pour l'embedding finetuning



### importation des librairies & fonctions

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import json
from datasets import DatasetDict, Dataset
import re
import numpy as np
from sentence_transformers import SentenceTransformer

import re

def remove_irrelevant_sections(description):
    """
    Supprime les sections non pertinentes telles que "About the Company," "Perks & Benefits,"
    et "Responsibilities" dans une description d'offre d'emplois.

    Args:
        description (str): La description du poste sous forme de chaîne de caractères.s a string.

    Returns:
        str: La description de poste nettoyée, dont les sections non pertinentes ont été supprimées.
    """
    # Définir des modèles d'expression régulière pour les sections à supprimer
    patterns = [
        r"(About the Company:|Our Mission:).*?(?=(Qualifications|Requirements|Skills|Experience|$))",
        r"(Perks & Benefits:|What We Offer:).*?(?=(Qualifications|Requirements|Skills|Experience|$))",
        r"(Responsibilities:).*?(?=(Qualifications|Requirements|Skills|Experience|$))"
    ]

    # Supprimer chaque motif
    for pattern in patterns:
        description = re.sub(pattern, "", description, flags=re.IGNORECASE | re.DOTALL)

    return description.strip()

def extract_qualifications_from_html(description):
    """
    Extraits de sections d'une description de poste qui commencent par des mots-clés tels que
    "Qualifications," "Requirements," "Skills," ou "Experience."

    Args:
        description (str): La description du poste sous forme de chaîne de caractères.

    Returns:
        str: La section pertinente contenant les qualifications, ou la description originale
             si aucune correspondance n'est trouvée.
    """
    # Rechercher des sections commençant par des mots-clés pertinents
    match = re.search(
        r"(Qualifications|Requirements|Skills|Experience).*",
        description,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if match:
        # Extraire la section correspondante
        relevant_section = match.group(0)
        return relevant_section
    return description

def remove_eoe_notes(description):
    """
    Supprime les mentions Equal Opportunity Employer (EOE)  et autres textes standard similaires
    de la description du poste.

    Args:
        description (str): La description du poste sous forme de chaîne de caractères.

    Returns:
        str: La description de poste nettoyée, sans les mentions EOE.
    """
    # Définir des modèles d'expressions régulières pour les notes EOE courantes
    patterns = [
        r"an equal opportunity employer.*?(?=\n|$)",  # Expressions courantes
        r"EOE.*?(?=\n|$)",  # Forme courte
        r"EEO.*?(?=\n|$)",
        r"equal employment*?(?=\n|$)",  # Modèle complet
        r"Equal employment opportunity.*?(?=\n|$)"  # Variations
    ]

    # Supprimer chaque motif de la description
    for pattern in patterns:
        description = re.sub(pattern, "", description, flags=re.IGNORECASE | re.DOTALL)

    return description.strip()


### load data (batch)

In [ ]:
# # extract JDs
# df_jobs = pd.read_csv("data/job_data.csv")
# # df_jobs = df_jobs.drop_duplicates()

# # only keep text relevant to job qualifications
# df_jobs['description_cleaned'] = df_jobs['description'].apply(remove_irrelevant_sections)
# df_jobs['description_cleaned'] = df_jobs['description_cleaned'].apply(extract_qualifications_from_html)
# df_jobs['description_cleaned'] = df_jobs['description_cleaned'].apply(remove_eoe_notes)

# # store job descriptions in a list
# job_description_list = df_jobs['description_cleaned'].to_list()

In [ ]:
# # extract synthetic queries and store in list (from batch request_
# file_path = 'data/output.jsonl'
# query_list = []

# with open(file_path, 'r') as file:
#     for line in file:
#         query = json.loads(line)['response']['body']['choices'][0]['message']['content'].replace('"', '')
#         query_list.append(query)

In [ ]:
# # create dict with queries and JDs
# df = pd.DataFrame({"query" : query_list, "job_description_pos" : job_description_list})

### Charger les données (à partir d'un csv)

In [2]:
# Extraire les descriptions de poste
df_jobs = pd.read_csv("job_data_w_query.csv")

# Ne conservez que les informations pertinentes pour les qualifications requises pour le poste.
df_jobs['job_description_pos'] = df_jobs['description'].apply(remove_irrelevant_sections)
df_jobs['job_description_pos'] = df_jobs['job_description_pos'].apply(extract_qualifications_from_html)
df_jobs['job_description_pos'] = df_jobs['job_description_pos'].apply(remove_eoe_notes)

# stocker les descriptions de poste dans une liste
df = df_jobs[['query', 'job_description_pos']]

In [3]:
# Supprimer les doublons
print("Forme d'origine:", df.shape)
df = df.drop_duplicates(subset=['job_description_pos'])
print("Description de postes uniques:", df.shape)
df = df.drop_duplicates(subset=['query'])
print("Requêtes uniques:",df.shape)

Original shape: (1100, 2)
Unique JDs: (947, 2)
Unique queries: (947, 2)


In [4]:
df.head()

,query,job_description_pos
0,"Kafka, Snowflake, Informatica IICS",experience neededVery strong experience in Kaf...
1,"Subaru R&D job search: automotive engineering,...",requirements to determine feasibility of desig...
2,Full stack developer React Node.js AWS Lambda,"experienceAccountable for code quality, includ..."
3,Data Analyst Queens NY contract data modeling ...,"QualificationsAnalytical skills, including the..."
4,"mortgage banking data systems, high-performanc...",requirements and industry practices for mortga...


### créer des paires négatives

In [5]:
# Charger le modèle
model = SentenceTransformer("all-mpnet-base-v2")

In [6]:
%%time
# Encoder toutes les descriptions de postes
job_embeddings = model.encode(df['job_description_pos'].to_list())
print(job_embeddings.shape)

(947, 768)
CPU times: user 17min 6s, sys: 3min 48s, total: 20min 55s
Wall time: 21min 6s


In [7]:
# Calculer les similarités sémantiques
similarities = model.similarity(job_embeddings, job_embeddings)
print(similarities.shape)

torch.Size([947, 947])


In [8]:
# correspondance la moins similaire entre les descriptions de poste  à correspondance positive comme correspondance négative
similarities_argsorted = np.argsort(similarities.numpy(), axis=1)
negative_pair_index_list = []

for i in range(len(similarities)):

    # Commencez par l'indice de similarité le plus faible pour la ligne actuelle.
    j = 0
    index = int(similarities_argsorted[i][j])

    # S'assurer que l'index est unique
    while index in negative_pair_index_list:
        j += 1  # Passez à l'index suivant le plus petit.
        index = int(similarities_argsorted[i][j])  # Récupérer l'index suivant le plus petit

    negative_pair_index_list.append(index)

In [9]:
# Ajouter des paires négatives à df
df['job_description_neg'] = df['job_description_pos'].iloc[negative_pair_index_list].values

In [10]:
df.head()

,query,job_description_pos,job_description_neg
0,"Kafka, Snowflake, Informatica IICS",experience neededVery strong experience in Kaf...,"qualifications, skills, competencies, competen..."
1,"Subaru R&D job search: automotive engineering,...",requirements to determine feasibility of desig...,SQL (expert)Snowflake - not a roadblock (added...
2,Full stack developer React Node.js AWS Lambda,"experienceAccountable for code quality, includ...",Resource should be able to visualize and expla...
3,Data Analyst Queens NY contract data modeling ...,"QualificationsAnalytical skills, including the...",experiences. We own and operate leading entert...
4,"mortgage banking data systems, high-performanc...",requirements and industry practices for mortga...,Qualifications:\nFluency in English (native or...


### train-eval-test split

In [11]:
# Mélanger l'ensemble de données
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Divisez en ensembles d'entraînement, de validation et de test (par exemple, 80 % d'entraînement, 10 % de validation, 10 % de test).
train_frac = 0.8
valid_frac = 0.1
test_frac = 0.1

# Définir la taille de l'échantillon d'entraînement et de validation
train_size = int(train_frac * len(df))
valid_size = int(valid_frac * len(df))

# Créer des ensembles de données d'entraînement, de validation et de test.
df_train = df[:train_size]
df_valid = df[train_size:train_size + valid_size]
df_test = df[train_size + valid_size:]

### Télécharger vers Hugging Face Hub

In [12]:
# Convertissez les DataFrames pandas en ensembles de données Hugging Face.
train_ds = Dataset.from_pandas(df_train)
valid_ds = Dataset.from_pandas(df_valid)
test_ds = Dataset.from_pandas(df_test)

# Combiner dans un DatasetDict
dataset_dict = DatasetDict({
    'train': train_ds,
    'validation': valid_ds,
    'test': test_ds
})

In [13]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['query', 'job_description_pos', 'job_description_neg'],
        num_rows: 757
    })
    validation: Dataset({
        features: ['query', 'job_description_pos', 'job_description_neg'],
        num_rows: 94
    })
    test: Dataset({
        features: ['query', 'job_description_pos', 'job_description_neg'],
        num_rows: 96
    })
})

In [14]:
# transférer les données vers le hub
dataset_dict.push_to_hub("Fe2x/ai-job-embedding-finetuning")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  49%|####8     | 1.05MB / 2.14MB            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  267kB /  267kB            

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Fe2x/ai-job-embedding-finetuning/commit/90f9c04c023dff67f05dfa4a5d5a99dd24996075', commit_message='Upload dataset', commit_description='', oid='90f9c04c023dff67f05dfa4a5d5a99dd24996075', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Fe2x/ai-job-embedding-finetuning', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Fe2x/ai-job-embedding-finetuning'), pr_revision=None, pr_num=None)